# Run Order Formatter

Converts any agility trial spreadsheet with the standard headers into a clean, printable PDF.

**Expected columns:** `Date`, `Event`, `Level`, `Class Type`, `Height`, `Handler Name`, `Dog`, `Breed`

**Features:**
- Grouped by Day → Class → Level
- Optional handler highlight (yellow rows)
- Gap analysis: how many runs separate a handler's consecutive entries in the same class

---
**How to use:** Edit the `Configuration` cell, then run all cells top-to-bottom (`Cell → Run All`).

In [33]:
%pip install pandas xlrd openpyxl reportlab --quiet

ERROR: Exception:
Traceback (most recent call last):
  File "/Users/victoriahenderson/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/Users/victoriahenderson/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/Users/victoriahenderson/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/req_command.py", line 96, in wrapper
    return func(self, options, args)
  File "/Users/victoriahenderson/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/commands/install.py", line 487, in run
    if summary := installed_packages_summary(installed, env):
  File "/Users/victoriahenderson/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/commands/install.py", line 636, in installed_packages_summary
    installed_versions[distribution.canonical_name] = distribution.version
  File "/Users/victoriahenderson

In [34]:
import pandas as pd
from pathlib import Path
from IPython.display import display

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle,
    Paragraph, Spacer, PageBreak, HRFlowable
)

In [35]:
# ── EDIT THESE ──────────────────────────────────────────────────
SPREADSHEET_PATH  = "grid_print_run_order.xls"   # relative to this notebook, or full path
OUTPUT_PDF        = "run_order.pdf"
HIGHLIGHT_HANDLER = None        # set to None to disable
TRIAL_TITLE       = "UKI Agility Trial — Run Order"

# Level grouping — add/remove level names to match your trial data
SC_LEVELS = frozenset({'champ', 'select', 'senior', 'regular', 'palladium'})
BN_LEVELS = frozenset({'novice', 'beginner'})

# Blocks smaller than this are considered part of an Open/interleaved class
# and won't get a "Senior / Champ" or "Beginner / Novice" sub-header.
# Increase this number if you're seeing unexpected sub-headers.
MIN_BLOCK_SIZE = 5
# ────────────────────────────────────────────────────────────────

In [36]:
def load_spreadsheet(path):
    path = Path(path)

    if path.suffix.lower() == ".xls":
        # Force xlrd's CompDoc to ignore minor OLE sector-map corruption.
        # CompDoc.__init__ accepts ignore_workbook_corruption but pandas < 1.5
        # has no way to pass it through, so we patch __init__ to always set it.
        from xlrd import compdoc
        _orig_init = compdoc.CompDoc.__init__

        def _lenient_init(self, *args, **kwargs):
            kwargs["ignore_workbook_corruption"] = True
            _orig_init(self, *args, **kwargs)

        compdoc.CompDoc.__init__ = _lenient_init
        try:
            df = pd.read_excel(path, engine="xlrd")
        finally:
            compdoc.CompDoc.__init__ = _orig_init   # always restore
    else:
        df = pd.read_excel(path, engine="openpyxl")

    df.columns = [str(c).strip() for c in df.columns]

    # Date and Event may only appear on the first row of each group — fill down
    for col in ["Date", "Event"]:
        if col in df.columns:
            df[col] = df[col].replace("", pd.NA).ffill()

    df.dropna(how="all", inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"Loaded {len(df)} rows  |  columns: {df.columns.tolist()}")
    return df

In [37]:
df = load_spreadsheet(SPREADSHEET_PATH)

print("\n── Sample rows ──")
display(df.head(10))

print("\n── Unique values ──")
for col in ["Date", "Event", "Level"]:
    if col in df.columns:
        print(f"{col:10}: {df[col].dropna().unique().tolist()}")

Loaded 313 rows  |  columns: ['Date', 'Event', 'Level', 'Class Type', 'Height', 'Handler Name', 'Dog', 'Breed']

── Sample rows ──


,Date,Event,Level,Class Type,Height,Handler Name,Dog,Breed
0,Saturday,Gamblers,Champ,Select,4,Kim Black,Caisson,Miniature American Shepherd
1,Saturday,Gamblers,Novice,Select,8,Paulena Renee Simpson,Wild Card,Border Collie/Toy Aussie
2,Saturday,Gamblers,Senior,Select,8,Scott Hudson,Theo,Crossbreed
3,Saturday,Gamblers,Novice,Regular,8,Bridget Isele,Bannah,Crossbreed
4,Saturday,Gamblers,Champ,Regular,8,Kim Black,Rip,Miniature American Shepherd
5,Saturday,Gamblers,Senior,Select,12,Donna VanWinkle,Wimsey,Miniature Poodle
6,Saturday,Gamblers,Champ,Select,16,Holly Hale,Suri,Border Collie
7,Saturday,Gamblers,Novice,Select,16,Lachelle Henderson,Tsunami,Belgian Malinois
8,Saturday,Gamblers,Novice,Select,20,Paulena Renee Simpson,Palladium,Border Collie
9,Saturday,Gamblers,Champ,Regular,20,Merritt Speagle,Surefire,Border Collie



── Unique values ──
Date      : ['Saturday', 'Sunday']
Event     : ['Gamblers', 'Agility', 'Snooker', 'Jumping', 'Speedstakes', 'Masters Jumping', 'Masters Agility']
Level     : ['Champ', 'Novice', 'Senior', 'Beginner', 'Masters']


In [38]:
def analyze_gaps(df, handler_name):
    """
    For each class the handler is entered in, shows how many runs
    separate each pair of their consecutive entries.

    Class = Date + Event (entire class, not split by level).
    'Runs Between' = entries by OTHER handlers between this handler's
    back-to-back appearances within the same class.
    """
    df = df.copy()
    df["_class"] = (
        df["Date"].fillna("").astype(str).str.strip() + " — " +
        df["Event"].fillna("").astype(str).str.strip()
    )

    name_lower = handler_name.strip().lower()
    rows = []

    for cls in df["_class"].unique():
        cls_df = df[df["_class"] == cls].reset_index(drop=True)
        mask   = cls_df["Handler Name"].str.strip().str.lower() == name_lower
        h      = cls_df[mask]

        if h.empty:
            continue

        positions = h.index.tolist()
        for i, pos in enumerate(positions):
            dog = str(h.loc[pos, "Dog"]).strip()
            if i < len(positions) - 1:
                nxt      = positions[i + 1]
                gap      = nxt - pos - 1
                next_dog = str(h.loc[nxt, "Dog"]).strip()
                rows.append({
                    "Class":         cls,
                    "Run #":         pos + 1,
                    "Dog":           dog,
                    "Runs Between":  gap,
                    "Next Run #":    nxt + 1,
                    "Next Dog":      next_dog,
                })
            else:
                rows.append({
                    "Class":         cls,
                    "Run #":         pos + 1,
                    "Dog":           dog,
                    "Runs Between":  "(last entry in class)",
                    "Next Run #":    "—",
                    "Next Dog":      "—",
                })

    if not rows:
        print(f"No runs found for '{handler_name}'.")
        return []

    gap_df = pd.DataFrame(rows)[["Class","Run #","Dog","Runs Between","Next Run #","Next Dog"]]
    print(f"\nGap Analysis — {handler_name}")
    print("=" * 80)
    display(gap_df)
    return rows


if HIGHLIGHT_HANDLER:
    gap_results = analyze_gaps(df, HIGHLIGHT_HANDLER)

In [ ]:
# ── Color palette ────────────────────────────────────────────────
C_DARK_BLUE  = colors.HexColor("#1E3A8A")
C_MED_BLUE   = colors.HexColor("#3B82F6")
C_HIGHLIGHT  = colors.HexColor("#FEF9C3")
C_ALT_ROW    = colors.HexColor("#EFF6FF")
C_SUBHEADER  = colors.HexColor("#BFDBFE")
C_GRID       = colors.HexColor("#CBD5E1")

GROUP_LABELS = {"SC": "Senior / Champ", "BN": "Beginner / Novice", "M": "Masters"}


def _level_group(level_str):
    l = str(level_str).strip().lower()
    if l in SC_LEVELS:  return "SC"
    if l in BN_LEVELS:  return "BN"
    if "masters" in l:  return "M"
    return "OTHER"


def _get_blocks(class_df):
    """Consecutive same-group runs → list of (group, start_idx, end_idx)."""
    groups = class_df["Level"].apply(_level_group).tolist()
    if not groups:
        return []
    blocks, start, cur = [], 0, groups[0]
    for i in range(1, len(groups)):
        if groups[i] != cur:
            blocks.append((cur, start, i))
            start, cur = i, groups[i]
    blocks.append((cur, start, len(groups)))
    return blocks


def _split_into_heats(blocks):
    """
    Split blocks into heats. A new heat begins when a *large* block's group
    has already appeared in the current heat — i.e. the SC→BN cycle repeats.
    Small blocks (< MIN_BLOCK_SIZE) never trigger a new heat.
    Returns a list of block-lists, one per heat.
    """
    if not any(b[2] - b[1] >= MIN_BLOCK_SIZE for b in blocks):
        return [blocks]  # all small → one Open heat, no split

    heats, current, seen = [], [], set()
    for grp, s, e in blocks:
        is_large = (e - s) >= MIN_BLOCK_SIZE
        if is_large and grp in seen:
            # This group already appeared → start a fresh heat
            heats.append(current)
            current, seen = [(grp, s, e)], {grp}
        else:
            current.append((grp, s, e))
            if is_large:
                seen.add(grp)
    if current:
        heats.append(current)
    return heats


def _make_table(table_data, col_widths, highlight_rows=(), subheader_rows=(), extra_cmds=()):
    cmds = [
        ("BACKGROUND",    (0, 0), (-1, 0), C_DARK_BLUE),
        ("TEXTCOLOR",     (0, 0), (-1, 0), colors.white),
        ("FONTNAME",      (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE",      (0, 0), (-1, 0), 10),
        ("ALIGN",         (0, 0), (-1, 0), "CENTER"),
        ("FONTNAME",      (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE",      (0, 1), (-1, -1), 9),
        ("GRID",          (0, 0), (-1, -1), 0.4, C_GRID),
        ("TOPPADDING",    (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("LEFTPADDING",   (0, 0), (-1, -1), 7),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 7),
        ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
    ]
    for i in range(1, len(table_data)):
        if i not in subheader_rows and i % 2 == 0:
            cmds.append(("BACKGROUND", (0, i), (-1, i), C_ALT_ROW))
    for i in subheader_rows:
        cmds += [
            ("SPAN",          (0, i), (-1, i)),
            ("BACKGROUND",    (0, i), (-1, i), C_SUBHEADER),
            ("TEXTCOLOR",     (0, i), (-1, i), C_DARK_BLUE),
            ("FONTNAME",      (0, i), (-1, i), "Helvetica-BoldOblique"),
            ("FONTSIZE",      (0, i), (-1, i), 9),
            ("ALIGN",         (0, i), (-1, i), "CENTER"),
            ("TOPPADDING",    (0, i), (-1, i), 3),
            ("BOTTOMPADDING", (0, i), (-1, i), 3),
        ]
    for i in highlight_rows:
        if i not in subheader_rows:
            cmds += [
                ("BACKGROUND", (0, i), (-1, i), C_HIGHLIGHT),
                ("FONTNAME",   (0, i), (-1, i), "Helvetica-Bold"),
                ("TEXTCOLOR",  (0, i), (-1, i), C_DARK_BLUE),
            ]
    cmds += list(extra_cmds)
    tbl = Table(table_data, colWidths=col_widths, repeatRows=1)
    tbl.setStyle(TableStyle(cmds))
    return tbl


def _single_heat_flowables(heat_df, heat_label, heat_blocks, highlight_handler, col_widths):
    """Build the header paragraph + table for one heat."""
    header_style = ParagraphStyle(
        "ClassH", fontSize=11, fontName="Helvetica-Bold",
        spaceBefore=14, spaceAfter=3, textColor=C_DARK_BLUE
    )
    n = sum(b[2] - b[1] for b in heat_blocks)
    label = f"{heat_label.strip().upper()}  ({n} run{'s' if n != 1 else ''})"

    large_in_heat = [b for b in heat_blocks if b[2] - b[1] >= MIN_BLOCK_SIZE]
    show_sh = len(large_in_heat) >= 2

    table_data = [["#", "Handler", "Dog", "Breed", "Height"]]
    hi_rows, sh_rows = set(), set()
    run_num = 1

    for grp, blk_s, blk_e in heat_blocks:
        is_large = (blk_e - blk_s) >= MIN_BLOCK_SIZE
        if show_sh and is_large:
            sh_rows.add(len(table_data))
            table_data.append([f"— {GROUP_LABELS.get(grp, grp)} —", "", "", "", ""])

        for i in range(blk_s, blk_e):
            row     = heat_df.iloc[i]
            handler = str(row.get("Handler Name", "")).strip()
            dog     = str(row.get("Dog",          "")).strip()
            breed   = str(row.get("Breed",        "")).strip()
            height  = str(row.get("Height",       "")).strip()
            if height == "nan":
                height = ""
            table_data.append([run_num, handler, dog, breed, height])
            if highlight_handler and handler.lower() == highlight_handler.strip().lower():
                hi_rows.add(len(table_data) - 1)
            run_num += 1

    return [
        Paragraph(label, header_style),
        _make_table(table_data, col_widths, highlight_rows=hi_rows, subheader_rows=sh_rows),
    ]


def _class_flowables(class_df, event, highlight_handler):
    """
    Return flowables for one event.
    Events with repeating SC/BN cycles (e.g. Agility, Jumping) are split into
    numbered sections: AGILITY, AGILITY 2, AGILITY 3 …
    Run numbers restart at 1 within each heat.
    """
    col_widths = [0.45*inch, 2.4*inch, 1.55*inch, 2.85*inch, 0.65*inch]
    class_df   = class_df.reset_index(drop=True)
    blocks     = _get_blocks(class_df)
    heats      = _split_into_heats(blocks)

    flowables = []
    for heat_idx, heat_blocks in enumerate(heats):
        suffix     = f" {heat_idx + 1}" if heat_idx > 0 else ""
        heat_label = f"{event.strip()}{suffix}"
        flowables.extend(
            _single_heat_flowables(class_df, heat_label, heat_blocks, highlight_handler, col_widths)
        )
        if heat_idx < len(heats) - 1:
            flowables.append(Spacer(1, 0.1 * inch))

    return flowables


def _gap_flowables(df, handler_name):
    title_style = ParagraphStyle(
        "GapT", fontSize=14, fontName="Helvetica-Bold",
        alignment=TA_CENTER, spaceAfter=6, textColor=C_DARK_BLUE
    )
    note_style = ParagraphStyle(
        "GapN", fontSize=9, fontName="Helvetica-Oblique",
        alignment=TA_CENTER, spaceAfter=14,
        textColor=colors.HexColor("#6B7280")
    )
    flowables = [
        PageBreak(),
        Paragraph(f"Run Gap Analysis — {handler_name}", title_style),
        Paragraph(
            '"Runs Between" = number of other handlers running between '
            "this handler's consecutive entries within the same class.",
            note_style
        ),
    ]
    df = df.copy()
    df["_class"] = (
        df["Date"].fillna("").astype(str).str.strip() + " — " +
        df["Event"].fillna("").astype(str).str.strip()
    )
    name_lower = handler_name.strip().lower()
    col_widths = [3.0*inch, 0.65*inch, 1.4*inch, 1.4*inch, 1.0*inch, 1.4*inch]
    table_data = [["Class", "Run #", "Dog", "Runs Between", "Next Run #", "Next Dog"]]
    for cls in df["_class"].unique():
        cls_df = df[df["_class"] == cls].reset_index(drop=True)
        mask   = cls_df["Handler Name"].str.strip().str.lower() == name_lower
        h      = cls_df[mask]
        if h.empty:
            continue
        positions = h.index.tolist()
        for i, pos in enumerate(positions):
            dog = str(h.loc[pos, "Dog"]).strip()
            if i < len(positions) - 1:
                nxt      = positions[i + 1]
                gap      = nxt - pos - 1
                next_dog = str(h.loc[nxt, "Dog"]).strip()
                table_data.append([cls, pos+1, dog, gap, nxt+1, next_dog])
            else:
                table_data.append([cls, pos+1, dog, "(last entry)", "—", "—"])
    if len(table_data) == 1:
        flowables.append(Paragraph(f"No runs found for '{handler_name}'.", note_style))
        return flowables
    tbl = _make_table(table_data, col_widths, extra_cmds=[
        ("ALIGN", (1, 1), (1, -1), "CENTER"),
        ("ALIGN", (3, 1), (4, -1), "CENTER"),
    ])
    flowables.append(tbl)
    return flowables


def _day_banner(date_val):
    data = [[f"  ◆   {str(date_val).upper()}   ◆  "]]
    tbl  = Table(data, colWidths=[9.9 * inch])
    tbl.setStyle(TableStyle([
        ("BACKGROUND",    (0, 0), (-1, -1), C_DARK_BLUE),
        ("TEXTCOLOR",     (0, 0), (-1, -1), colors.white),
        ("FONTNAME",      (0, 0), (-1, -1), "Helvetica-Bold"),
        ("FONTSIZE",      (0, 0), (-1, -1), 13),
        ("ALIGN",         (0, 0), (-1, -1), "CENTER"),
        ("TOPPADDING",    (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))
    return tbl


def generate_pdf(df, output_path, highlight_handler=None, title=None):
    if title is None:
        title = TRIAL_TITLE
    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=landscape(letter),
        rightMargin=0.5 * inch, leftMargin=0.5 * inch,
        topMargin=0.55 * inch,  bottomMargin=0.55 * inch,
    )
    title_style = ParagraphStyle(
        "Title", fontSize=18, fontName="Helvetica-Bold",
        alignment=TA_CENTER, spaceAfter=4, textColor=C_DARK_BLUE
    )
    story = [
        Paragraph(title, title_style),
        HRFlowable(width="100%", thickness=1.5, color=C_MED_BLUE, spaceAfter=8),
    ]
    for date_val in df["Date"].dropna().unique():
        date_df = df[df["Date"] == date_val]
        story.append(Spacer(1, 0.1 * inch))
        story.append(_day_banner(date_val))
        for event, class_df in date_df.groupby("Event", sort=False):
            story.extend(_class_flowables(class_df, str(event), highlight_handler))
            story.append(Spacer(1, 0.1 * inch))
    if highlight_handler:
        story.extend(_gap_flowables(df, highlight_handler))
    doc.build(story)
    print(f"PDF saved → {output_path}")

In [40]:
output = Path(SPREADSHEET_PATH).parent / OUTPUT_PDF

generate_pdf(df, output, highlight_handler=HIGHLIGHT_HANDLER)

print(f"\nDone! File saved to: {output.resolve()}")

PDF saved → run_order.pdf

Done! File saved to: /Users/victoriahenderson/Documents/GitHub/personal_projects/FormattingTools/run_order.pdf
